# 🧠 AI Governance & Monitoring — Demo Notebook

This notebook demonstrates:
- AI model prediction (GPT-J small)
- MLflow experiment tracking
- Evidently drift detection
- SHAP global explainability
- LIME local explainability
- OPA policy enforcement
- Lineage logging

This is based on your full open-source monitoring + governance repo scaffold.

## 1️⃣ Install Dependencies

In [ ]:
!pip install transformers torch mlflow evidently shap lime openlineage-python pandas matplotlib scikit-learn

## 2️⃣ Load Sample Model (Small HF model for speed)

In [ ]:
from transformers import pipeline

model = pipeline("text-generation", model="distilgpt2")
print("Model Loaded ✔")

## 3️⃣ Run a Prediction

In [ ]:
prompt = "AI governance requires transparency in model decisions because"
response = model(prompt, max_length=60)[0]['generated_text']
response

## 4️⃣ Log Run in MLflow (Governance Metadata)

In [ ]:
import mlflow

mlflow.set_experiment("AI_Governance_Demo")

with mlflow.start_run():
    mlflow.log_param("model", "distilgpt2")
    mlflow.log_param("task", "text-generation")
    mlflow.log_metric("response_length", len(response))
    mlflow.log_text(response, "output.txt")

print("MLflow logging complete ✔")

## 5️⃣ Drift Detection with Evidently

In [ ]:
import pandas as pd
from evidently.report import Report
from evidently.metric_preset import DataDriftPreset

# Load reference + current datasets
ref = pd.read_csv("data/reference.csv")
curr = pd.read_csv("data/current.csv")

report = Report(metrics=[DataDriftPreset()])
report.run(reference_data=ref, current_data=curr)
report.save_html("dashboards/demo_drift_report.html")

"Drift report generated → dashboards/demo_drift_report.html"

## 6️⃣ SHAP Explainability (Global Feature Importance)

In [ ]:
import shap
from sklearn.ensemble import RandomForestClassifier
import matplotlib.pyplot as plt

df = pd.read_csv("explainability/sample_data.csv")
X = df.drop("target", axis=1)
y = df["target"]

model_rf = RandomForestClassifier().fit(X, y)

explainer = shap.TreeExplainer(model_rf)
shap_values = explainer.shap_values(X)

shap.summary_plot(shap_values, X, show=False)
plt.savefig("dashboards/shap_summary_demo.png")

"SHAP summary saved → dashboards/shap_summary_demo.png"

## 7️⃣ LIME Explainability (Local Instance Explanation)

In [ ]:
from lime.lime_tabular import LimeTabularExplainer

explainer_lime = LimeTabularExplainer(
    training_data=X.values,
    feature_names=X.columns,
    class_names=['0','1'],
    mode='classification'
)

exp = explainer_lime.explain_instance(
    X.iloc[0].values,
    model_rf.predict_proba
)

exp.save_to_file("dashboards/lime_demo.html")
"LIME explanation saved → dashboards/lime_demo.html"

## 8️⃣ OPA Policy Enforcement (Governance Rules)

In [ ]:
import json
import subprocess

policy = "governance/opa_policy.rego"
input_file = "governance/policy_test.json"

result = subprocess.run([
    "opa", "eval", "--format=json", "--data", policy, "--input", input_file, "data.ai.policy.allow"
], capture_output=True, text=True)

print("OPA Decision: ", result.stdout)

## 9️⃣ Lineage Logging (OpenLineage)

In [ ]:
from openlineage.client.run import RunEvent, RunState
from datetime import datetime

event = RunEvent(
    eventType=RunState.COMPLETE,
    eventTime=datetime.utcnow().isoformat(),
    run={"runId": "demo-run-001"},
    job={"namespace": "ai-governance", "name": "demo-job"},
)

print("Lineage event created: ", event.to_dict())

# 🎉 Demo Completed
You now demonstrated:
- Model inference
- MLflow governance logging
- Data drift monitoring (Evidently)
- Model explainability (SHAP + LIME)
- Policy enforcement (OPA)
- Lineage metadata logging

This fully aligns with your thesis: **Monitoring & Governance of AI Models, Solutions & Agents using Open-Source Tools.**